## Proyecto Recomendador de Películas Similares

Vamos a construir un recomendador de de películas usando una base real CSV y luego lo convertiremos en una base vectorial para finalizar con un RAG de películas

In [ ]:
import os
import ast

from pathlib import Path
from pinecone import Pinecone, ServerlessSpec
from dotenv import load_dotenv
from langchain_ollama import ChatOllama, OllamaEmbeddings
from langchain_pinecone import PineconeVectorStore
from langchain_core.documents import Document
import pandas as pd

In [2]:

#* Configuración urls y carga de variables de entorno
load_dotenv()

PINECONE_API_KEY = os.getenv("PINECONE_API_KEY")

BASE_DIR = Path.cwd().parent.parent
DATA_DIR = BASE_DIR / "assets"
csv_peliculas_path =  DATA_DIR / "datos" / "movies_metadata.csv"

### Limpieza, reducción de datos y generación de documento

In [3]:
df_movies = pd.read_csv(str(csv_peliculas_path), low_memory=False)
df_movies = df_movies[df_movies['original_language'] == 'es']
df_movies['year'] = pd.to_datetime(df_movies['release_date'], errors='coerce').dt.year
df_movies = df_movies[df_movies['year'] >= 2010]
df_movies = df_movies[df_movies['adult'] != True]
df_movies = df_movies[df_movies['overview'].notnull()]

print(f"DataFrame de películas cargado con {len(df_movies)} registros.")

DataFrame de películas cargado con 367 registros.


In [4]:
def doc_construction(doc):
    doc_movie = f"""Título: {doc["title"]} ({int(doc["year"])}). Estado: {doc["status"]}.
    Sinopsis: {doc["overview"]}
    Detalles:
        - Título original: {doc["original_title"]}
        - Fecha de lanzamiento: {doc["release_date"]}
        - Géneros: {doc["genres"]}
        - Compañías de producción: {doc["production_companies"] if doc["production_companies"] else "N/A"}
        - Países de producción: {doc["production_countries"] if doc["production_countries"] else "N/A"}
        - Idiomas hablados: {doc["spoken_languages"]}
        - Duración: {int(doc["runtime"])} minutos.
        - Popularidad: {doc["popularity"]}.
        - Calificación promedio: {doc["vote_average"]} (sobre {int(doc["vote_count"])} votos).
    """
    return doc_movie

In [5]:
df_movies['genres'] = df_movies['genres'].apply(lambda x: ', '.join([i['name'] for i in ast.literal_eval(x)]) if pd.notnull(x) else '')
df_movies['production_companies'] = df_movies['production_companies'].apply(lambda x: ', '.join([i['name'] for i in ast.literal_eval(x)]) if pd.notnull(x) else '')
df_movies['production_countries'] = df_movies['production_countries'].apply(lambda x: ', '.join([i['name'] for i in ast.literal_eval(x)]) if pd.notnull(x) else '')
df_movies['spoken_languages'] = df_movies['spoken_languages'].apply(lambda x: ', '.join([i['iso_639_1'] for i in ast.literal_eval(x)]) if pd.notnull(x) else '')
df_movies['document'] = df_movies.apply(doc_construction, axis=1)

### Creación de Embeddings y Guardar en Pinecone

In [6]:

#* Configuración Pinecone
INDEX_NAME = "movies-db"
NAMESPACE = "movies-namespace"

pinecone_client = Pinecone(api_key=PINECONE_API_KEY)

In [7]:
existing_indexes = [idx["name"] for idx in pinecone_client.list_indexes()]

if INDEX_NAME not in existing_indexes:
    print(f"Creando indice '{INDEX_NAME}' en Pinecone...")
    pinecone_client.create_index(
        name=INDEX_NAME,
        dimension=768,
        metric="cosine",
        spec=ServerlessSpec(
            cloud="aws",
            region="us-east-1",
        ),
    )
    print("Índice creado.")
else:
    print(f"El índice '{INDEX_NAME}' ya existe en Pinecone.")

Creando indice 'movies-db' en Pinecone...
Índice creado.


In [8]:
pinecone_index = pinecone_client.Index(INDEX_NAME)
embedding_model = OllamaEmbeddings(model="embeddinggemma:300m")

In [9]:
documents = []

for _, row in df_movies.iterrows():
    metadata = {
        "id": str(row['id']),
        "title": row['title'] or row['original_title'],
        "original_title": row['original_title'],
        "year": str(int(row['year'])) if not pd.isnull(row['year']) else "N/A",
        "release_date": row['release_date'] or "N/A",
        "genres": row['genres'] or "N/A",
        "companies": row['production_companies'] or "N/A",
        "countries": row['production_countries'] or "N/A",
        "runtime": str(int(row['runtime'])) if not pd.isnull(row['runtime']) else "N/A",
        "popularity": str(row['popularity']) if not pd.isnull(row['popularity']) else "N/A",
        "vote_average": str(row['vote_average']) if not pd.isnull(row['vote_average']) else "N/A",
        "vote_count": str(int(row['vote_count'])) if not pd.isnull(row['vote_count']) else "N/A",
    }
    doc = Document(page_content=row['document'], metadata=metadata)
    
    documents.append(doc)

print(f"Total de documentos a indexar: {len(documents)}")

Total de documentos a indexar: 367


In [10]:
docsearch = PineconeVectorStore.from_documents(
    documents,
    embedding=embedding_model,
    index_name = INDEX_NAME,
    namespace=NAMESPACE
)

print("Documentos indexados en Pinecone con éxito.")

Index host ignored when initializing with index object.


Documentos indexados en Pinecone con éxito.


In [11]:
stats = pinecone_index.describe_index_stats()

dimensions = stats.get("dimension", "N/A")
metric = stats.get("metric", "N/A")
total_vectors = stats.get("total_vector_count", 0)
vector_type = stats.get("vector_type", "N/A")
namespaces = stats.get("namespaces", {})

print(f"=== Resumen del índice '{INDEX_NAME}' ===")
print(f"Dimensiones: {dimensions}")
print(f"Métrica: {metric}")
print(f"Tipo de vector: {vector_type}")
print(f"Total de vectores: {total_vectors}")
print(f"Namespaces: {list(namespaces.keys())}")
print("\nDetalle por namespace:")
for ns, data in namespaces.items():
    count = data.get("vector_count", 0)
    print(f" -'{ns}': {count} vectores")

=== Resumen del índice 'movies-db' ===
Dimensiones: 768
Métrica: cosine
Tipo de vector: dense
Total de vectores: 367
Namespaces: ['movies-namespace']

Detalle por namespace:
 -'movies-namespace': 367 vectores


### Creación de RAG para recomendación de películas

In [12]:
llm_model = ChatOllama(
    model="gemma3:4b",
    temperature=0.1
)

In [13]:
def search_movie(title_ref: str, df_movies: pd.DataFrame):
    
    title_ref_norm = title_ref.lower().strip()
    
    mask_exact = df_movies['title'].str.lower() == title_ref_norm
    if mask_exact.any():
        return df_movies[mask_exact].iloc[0]
    
    mask_partial = (
        df_movies['title'].str.contains(title_ref_norm, case=False, na=False) |
        df_movies['original_title'].str.contains(title_ref_norm, case=False, na=False)
    )
    
    if mask_partial.any():
        candidates = df_movies[mask_partial].sort_values("vote_count", ascending=False)
        return candidates.iloc[0]

    return None
    

In [20]:
def recommend_movies(title_ref: str, k: int = 5, explicar: bool = True, df_movies=df_movies):
    
    movie_ref = search_movie(title_ref, df_movies)
    
    if movie_ref is None:
        print(f"No se encontró la película con el título '{title_ref}'.")
        return

    try:
        year_ref = int(movie_ref['year'])
    except Exception:
        year_ref = movie_ref.get('year', None)
        
    generes_ref = movie_ref['genres']
    overview_ref = movie_ref['overview']
    
    print(f"Película de referencia: {movie_ref['title']} ({year_ref})")
    print(f"Géneros: {generes_ref}")
    print("-"*80)
    
    query_text = movie_ref['document']
    
    #* Búsqueda en la base vectorial
    resultados = docsearch.similarity_search_with_score(
        query_text,
        k=k+1,
        namespace=NAMESPACE
    )
    
    recomendaciones = []
    ref_id = str(movie_ref['id'])
    
    for doc, score in resultados:
        if doc.metadata.get("id") == ref_id:
            continue
        recomendaciones.append((doc, score))
        if len(recomendaciones) >= k:
            break
    
    if not recomendaciones:
        print("No se encontraron recomendaciones.")
        return
    
    
    print(f"Top {k} películas similares:\n")
    
    for idx, (doc, score) in enumerate(recomendaciones, start=1):
        meta = doc.metadata
        
        year = meta.get("year", "N/A")
        try:
            year = int(float(year))
        except Exception:
            year = "¿?"
        
        print(f"{idx}. {meta.get('title') if meta.get('title') else meta.get('original_title')} ({year}) - Score: {score:.4f}")
        print(f"   Géneros: {meta.get('genres', 'N/A')}")
        
        va = meta.get("vote_average", "N/A")
        if va is not None:
            try:
                vc = int(meta.get("vote_count", "N/A") or 0)
            except Exception:
                vc = meta.get("vote_count", "N/A")
            print(f"   Calificación Promedio: {va} (sobre {vc} votos)")
        
        print()
    
    if not explicar:
        return

    best_doc, best_score = recomendaciones[0]
    best_meta = best_doc.metadata
    
    best_overview = ""
    try:
        best_id = int(best_meta.get("id", "-1"))
        fila_movie = df_movies[df_movies['id'] == best_id].iloc[0]
        if not fila_movie.empty:
            best_overview = str(fila_movie.iloc[0].get('overview', '')).strip()
    except Exception:
        pass
    
    generes_best = best_meta.get("genres", "N/A")
    
    try:
        year_best = int(best_meta.get("year", "N/A"))
    except Exception:
        year_best = best_meta.get("year", "N/A")
    
    PROMPT_EXPLANATION = f"""
    Actúa como un cinéfilo que recomienda películas a otra persona.
    
    Quiero que expliques, en un párrafo corto y claro, por qué la siguiente película es una buena opción para alguien que disfrutó la película de referencia. Habla de temas, tono, género, época, estilo de historia, etc. NO menciones distancia coseno, embeddings, espacios vectoriales ni nada técnico.
    
    Película de referencia:
    - Título: {movie_ref['title']} ({year_ref})
    - Géneros: {generes_ref}
    
    - Sinopsis: {overview_ref}
    
    Película recomendada:
    - Título: {best_meta.get('title', 'N/A')} ({year_best})
    - Géneros: {generes_best}
    - Sinopsis: {best_overview}
    
    Escribe la explicación en segunda persona ("creo que te puede gustar esta película porque ..."), en español, en 4 a 6 líneas.
    """
    
    
    respuesta = llm_model.invoke(PROMPT_EXPLANATION)
    comentario = respuesta.content if hasattr(respuesta, 'content') else str(respuesta)
    print("Explicación de la recomendación:")
    print(comentario.strip())
    

In [24]:
def details_recommendation(title_ref: str, k: int = 5, n_chars: int = 600, df_movies=df_movies):
    
    movie_ref = search_movie(title_ref, df_movies)
    
    if movie_ref is None:
        print(f"No se encontró la película con el título '{title_ref}'.")
        return

    query_text = movie_ref['document']
    resultados = docsearch.similarity_search_with_score(
        query_text,
        k=k+1,
        namespace=NAMESPACE
    )
    
    ref_id = str(movie_ref['id'])
    recomendaciones = []
    for doc, score in resultados:
        if doc.metadata.get("id") == ref_id:
            ref_doc = doc
            continue
        recomendaciones.append((doc, score))
    
    if not recomendaciones:
        print("No se encontraron recomendaciones.")
        return
    
    best_doc, best_score = recomendaciones[0]
    ref_meta = ref_doc.metadata
    best_meta = best_doc.metadata
    
    year_ref = int(ref_meta.get("year", "N/A"))
    year_best = int(best_meta.get("year", "N/A"))
    
    generes_ref = ref_meta.get("genres", "N/A")
    generes_best = best_meta.get("genres", "N/A")
    
    arr_genres_ref = [g.strip().lower() for g in generes_ref.split(",")]
    arr_genres_best = [g.strip().lower() for g in generes_best.split(",")]
    
    common_genres = set(arr_genres_ref).intersection(set(arr_genres_best))
    
    print("\n=== Resumen numérico de la recomendación principal ===")
    print(f"- Película de referencia: {ref_meta.get('title', 'N/A')} ({year_ref})")
    print(f"- Película recomendada: {best_meta.get('title', 'N/A')} ({year_best})")
    print(f"- Géneros comunes: {', '.join(common_genres) if common_genres else 'Ninguno'}")
    print(f"- Score de similitud (distancia coseno): {best_score:.4f}")

In [25]:
title_ref = "Alexia"

recommend_movies(title_ref=title_ref, k=5, explicar=True)

Película de referencia: Alexia (2013)
Géneros: Horror, Thriller
--------------------------------------------------------------------------------
Top 5 películas similares:

1. For Elise (2013) - Score: 0.5464
   Géneros: Horror, Mystery
   Calificación Promedio: 4.8 (sobre 13 votos)

2. The 7th Floor (2013) - Score: 0.5401
   Géneros: Thriller
   Calificación Promedio: 5.8 (sobre 57 votos)

3. Mía (2011) - Score: 0.5118
   Géneros: Drama, Comedy
   Calificación Promedio: 7.7 (sobre 3 votos)

4. Death in Buenos Aires (2014) - Score: 0.4964
   Géneros: Crime, Drama, Mystery, Romance, Thriller
   Calificación Promedio: 5.7 (sobre 7 votos)

5. Absent (2011) - Score: 0.4944
   Géneros: Drama, Thriller
   Calificación Promedio: 6.6 (sobre 19 votos)

Explicación de la recomendación:
Creo que te puede gustar esta película porque, al igual que Alexia, *For Elise* te sumerge en una atmósfera de inquietud y misterio que te va a dejar sin aliento. Ambos son thrillers psicológicos con un tono muy p

In [26]:
details_recommendation(title_ref=title_ref, k=5, n_chars=600)


=== Resumen numérico de la recomendación principal ===
- Película de referencia: Alexia (2013)
- Película recomendada: For Elise (2013)
- Géneros comunes: horror
- Score de similitud (distancia coseno): 0.5464
